# 01 — Adopt a Construction Trade (`google/gemma-4-31B-it` + MutinyBIM RLVR)

Train a composable, spec-compliant **Trade LoRA** (~50 MB) for [`permitzip/mutinybim`](https://github.com/permitzip/mutinybim) using your Google Colab credits and push it to Hugging Face so anyone can blend it in the **Weight-Space Mixer**!

### Hardware & Credit Budget
* **Recommended Colab GPU:** **A100 (40GB or 80GB)** (~7–9 CCUs/hr). A 4-bit QLoRA SFT + focused GRPO pass typically takes **1.5–4 hours (~15–35 CCUs)**.
* **Pinned Weight-Space Contract (`spec.json`):**
  * Base model: `google/gemma-4-31B-it` (`unsloth/gemma-4-31B-it`)
  * LoRA: `r=16`, `lora_alpha=16`, `target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]`
  * MutinyBIM commit: `fbf5f61424b64d9f99b1c896ca02328c88d9813c`


In [ ]:
#@title 1. Check GPU, Clone Repos, Install Node 22, and Boot Warm MutinyBIM Verifier
!nvidia-smi

import os, subprocess, time, sys
from pathlib import Path

MIXER_REPO_URL = "https://github.com/5hyycjb77y/mutinybim-mixer.git" #@param {type:"string"}
WORK_DIR = Path("/content/work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
%cd /content/work

if not (WORK_DIR / "mutinybim-mixer").exists():
    # If you uploaded mutinybim-mixer locally or forked it, update MIXER_REPO_URL above.
    !git clone --depth 1 {MIXER_REPO_URL} mutinybim-mixer || true

MIXER_DIR = WORK_DIR / "mutinybim-mixer"
assert MIXER_DIR.exists(), "Clone failed — upload mutinybim-mixer to /content/work/mutinybim-mixer or set MIXER_REPO_URL."

import json
SPEC = json.loads((MIXER_DIR / "spec.json").read_text())
MUTINY_COMMIT = SPEC["mutinybim"]["commit"]
MUTINY_DIR = WORK_DIR / "mutinybim"

if not MUTINY_DIR.exists():
    !git clone {SPEC["mutinybim"]["repo"]} {MUTINY_DIR}
!git -C {MUTINY_DIR} fetch --depth 1 origin {MUTINY_COMMIT} && git -C {MUTINY_DIR} checkout {MUTINY_COMMIT}

# Install Node 22+ if Colab doesn't have it yet
node_v = subprocess.run(["node", "-v"], capture_output=True, text=True)
if node_v.returncode != 0 or int(node_v.stdout.strip().lstrip("v").split(".")[0]) < 22:
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs
!node -v

if not (MUTINY_DIR / "node_modules").exists():
    !npm --prefix {MUTINY_DIR} install --no-audit --no-fund

# Start persistent warm MutinyBIM verifier daemon
verifier_log = open("/content/verifier.log", "w")
verifier_proc = subprocess.Popen(
    ["node", str(MIXER_DIR / "verifier/server.mjs"), "--mutinybim", str(MUTINY_DIR), "--port", "8787"],
    stdout=verifier_log,
    stderr=subprocess.STDOUT,
)

sys.path.insert(0, str(MIXER_DIR))
from verifier.verify_client import MutinyVerifierClient

client = MutinyVerifierClient("http://127.0.0.1:8787")
for _ in range(30):
    try:
        health = client.health()
        print("Verifier ready:", health)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Verifier failed to start. Check /content/verifier.log")


In [ ]:
#@title 2. Install Unsloth, TRL, PEFT, and Hub Dependencies
!pip install -q --upgrade pip
!pip install -q unsloth trl peft accelerate bitsandbytes safetensors datasets huggingface_hub


In [ ]:
#@title 3. Contributor Configuration (Pick Your Trade!)
TRADE = "mechanical" #@param ["architecture", "mechanical", "plumbing", "electrical", "drafting", "structural", "fire_protection", "clash_negation", "coord_rlvr"]
RUN_SFT = True #@param {type:"boolean"}
RUN_GRPO = True #@param {type:"boolean"}
MAX_SFT_STEPS = 80 #@param {type:"integer"}
MAX_GRPO_STEPS = 120 #@param {type:"integer"}
LEARNING_RATE_SFT = 2e-4 #@param {type:"number"}
LEARNING_RATE_GRPO = 5e-6 #@param {type:"number"}
NUM_GENERATIONS = 4 #@param {type:"integer"}
ENABLE_THINKING = True #@param {type:"boolean"}
REJECTION_SAMPLE_COUNT = 2 #@param {type:"integer"}
HF_OUTPUT_REPO = "your-username/mutinybim-gemma4-31b-mechanical-v1" #@param {type:"string"}
PUSH_TO_HUB = False #@param {type:"boolean"}

OUTPUT_DIR = Path(f"/content/outputs/{TRADE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Configured trade={TRADE}, SFT={RUN_SFT} ({MAX_SFT_STEPS} steps), GRPO={RUN_GRPO} ({MAX_GRPO_STEPS} steps)")


In [ ]:
#@title 4. Load Live @permitzip/bim-script Reference & Official Examples
from mixer.prompts import TRADE_TASKS, MULTI_TRADE_BENCHMARK, format_task_prompt

ref_bundle = client.reference(compact=True)
KIT_REF_MD = ref_bundle["markdown"]
OFFICIAL_EXAMPLES = {ex["name"]: ex for ex in ref_bundle["examples"]}
print(f"Loaded reference ({len(KIT_REF_MD)} chars) and {len(OFFICIAL_EXAMPLES)} official examples:", list(OFFICIAL_EXAMPLES.keys()))

if TRADE in TRADE_TASKS:
    EVAL_TASKS = TRADE_TASKS[TRADE]
elif TRADE == "coord_rlvr":
    EVAL_TASKS = MULTI_TRADE_BENCHMARK
else:
    # clash_negation trains across MEP + Arch prompts
    EVAL_TASKS = [t for k in ("architecture", "mechanical", "plumbing", "electrical") for t in TRADE_TASKS[k]]

print(f"Active task pool for {TRADE}: {len(EVAL_TASKS)} tasks")


In [ ]:
#@title 5. Load unsloth/gemma-4-31B-it & Attach Spec-Compliant LoRA
import torch
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=SPEC["base_model"]["unsloth_id"],
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastModel.get_peft_model(
    model,
    r=SPEC["lora"]["r"],
    target_modules=SPEC["lora"]["target_modules"],
    lora_alpha=SPEC["lora"]["lora_alpha"],
    lora_dropout=SPEC["lora"]["lora_dropout"],
    bias=SPEC["lora"]["bias"],
    use_gradient_checkpointing="unsloth",
)
model.print_trainable_parameters()


In [ ]:
#@title 6. Build SFT Dataset (Official Examples + Self-Verified Rollouts)
from datasets import Dataset
from verifier.verify_client import extract_typescript

def render_chat(messages, add_generation_prompt=False):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=ENABLE_THINKING,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

sft_rows = []
# Seed from matching official MutinyBIM example(s)
trade_example = SPEC["trades"].get(TRADE, {}).get("example")
for ex_name, ex in OFFICIAL_EXAMPLES.items():
    if trade_example and ex_name != trade_example and TRADE not in ("coord_rlvr", "clash_negation"):
        continue
    msgs = format_task_prompt({"prompt": f"Write a complete MutinyBIM script: {ex['builds']}"}, KIT_REF_MD[:5000])
    msgs.append({"role": "assistant", "content": f"```ts\n{ex['source'].strip()}\n```"})
    sft_rows.append({"text": render_chat(msgs, add_generation_prompt=False)})

# Optional rejection sampling with Gemma 4 + warm MutinyBIM verifier
if REJECTION_SAMPLE_COUNT > 0:
    FastModel.for_inference(model)
    gen_cfg = SPEC["base_model"]["generation_defaults"]
    for task in EVAL_TASKS:
        msgs = format_task_prompt(task, KIT_REF_MD[:4500])
        prompt_text = render_chat(msgs, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        for _ in range(REJECTION_SAMPLE_COUNT):
            with torch.no_grad():
                out_ids = model.generate(
                    **inputs,
                    max_new_tokens=1536,
                    temperature=gen_cfg["temperature"],
                    top_p=gen_cfg["top_p"],
                    top_k=gen_cfg["top_k"],
                    do_sample=True,
                )
            comp = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
            ts = extract_typescript(comp)
            vr = client.verify(ts, start=task.get("start", "empty"), expected_kinds=task.get("expected_kinds"))
            keep = (vr.ok and vr.reward >= 0.85) if TRADE != "clash_negation" else (vr.breakdown.get("kernel_check_ok", 0) == 1.0 and vr.lint_summary.get("errors", 0) > 0)
            if keep:
                full_msgs = msgs + [{"role": "assistant", "content": f"```ts\n{ts}\n```"}]
                sft_rows.append({"text": render_chat(full_msgs, add_generation_prompt=False)})

if not sft_rows:
    # Guarantee non-empty SFT dataset from official examples
    for ex in OFFICIAL_EXAMPLES.values():
        msgs = format_task_prompt({"prompt": f"Build: {ex['builds']}"}, KIT_REF_MD[:5000])
        msgs.append({"role": "assistant", "content": f"```ts\n{ex['source'].strip()}\n```"})
        sft_rows.append({"text": render_chat(msgs, add_generation_prompt=False)})

sft_dataset = Dataset.from_list(sft_rows)
print(f"Prepared {len(sft_dataset)} verified SFT examples.")


In [ ]:
#@title 7. Stage A: SFT Warmup (SFTTrainer)
from trl import SFTTrainer, SFTConfig

if RUN_SFT:
    FastModel.for_training(model)
    sft_trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=sft_dataset,
        dataset_text_field="text",
        max_seq_length=4096,
        args=SFTConfig(
            output_dir=str(OUTPUT_DIR / "sft"),
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            max_steps=MAX_SFT_STEPS,
            learning_rate=LEARNING_RATE_SFT,
            logging_steps=5,
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            optim="adamw_8bit",
            save_strategy="no",
            report_to="none",
        ),
    )
    sft_trainer.train()
    sft_save = OUTPUT_DIR / "sft_adapter"
    model.save_pretrained(str(sft_save))
    tokenizer.save_pretrained(str(sft_save))
    print("Saved SFT adapter to", sft_save)


In [ ]:
#@title 8. Stage B: RLVR via GRPOTrainer Against MutinyBIM (bim-script + bim-lint + bim-calc)
from trl import GRPOTrainer, GRPOConfig
from verifier.verify_client import make_grpo_reward_func

if RUN_GRPO:
    FastModel.for_training(model)
    grpo_rows = []
    for task in EVAL_TASKS:
        msgs = format_task_prompt(task, KIT_REF_MD[:4000])
        grpo_rows.append({
            "prompt": render_chat(msgs, add_generation_prompt=True),
            "start": task.get("start", "empty"),
            "expected_kinds": task.get("expected_kinds", {}),
            "min_elements": task.get("min_elements", 1),
        })
    grpo_dataset = Dataset.from_list(grpo_rows)

    reward_fn = make_grpo_reward_func(
        client,
        default_start="empty",
        invert_for_clash_adapter=(TRADE == "clash_negation"),
    )

    grpo_args = GRPOConfig(
        output_dir=str(OUTPUT_DIR / "grpo"),
        learning_rate=LEARNING_RATE_GRPO,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_generations=NUM_GENERATIONS,
        max_prompt_length=2048,
        max_completion_length=1536,
        max_steps=MAX_GRPO_STEPS,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=1,
        save_strategy="no",
        report_to="none",
    )

    grpo_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[reward_fn],
        args=grpo_args,
        train_dataset=grpo_dataset,
    )
    grpo_trainer.train()

final_adapter_dir = OUTPUT_DIR / "final_adapter"
model.save_pretrained(str(final_adapter_dir))
tokenizer.save_pretrained(str(final_adapter_dir))
print("Saved final LoRA adapter to", final_adapter_dir)


In [ ]:
#@title 9. Standardized Eval Scorecard & Optional Push to Hugging Face Hub
FastModel.for_inference(model)
gen_cfg = SPEC["base_model"]["generation_defaults"]
eval_results = []

for task in EVAL_TASKS:
    msgs = format_task_prompt(task, KIT_REF_MD[:4000])
    prompt_text = render_chat(msgs, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=1536,
            temperature=0.6,
            top_p=gen_cfg["top_p"],
            top_k=gen_cfg["top_k"],
            do_sample=True,
        )
    comp = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    ts = extract_typescript(comp)
    vr = client.verify(
        ts,
        start=task.get("start", "empty"),
        expected_kinds=task.get("expected_kinds"),
        min_elements=task.get("min_elements"),
    )
    eval_results.append({
        "task_id": task["id"],
        "ok": vr.ok,
        "reward": vr.reward,
        "lint_errors": vr.lint_summary.get("errors", 0),
        "stage_failed": vr.stage_failed,
    })

n = max(len(eval_results), 1)
scorecard = {
    "pass_rate": round(sum(1 for r in eval_results if r["ok"]) / n, 4),
    "zero_clash_rate": round(sum(1 for r in eval_results if r["lint_errors"] == 0) / n, 4),
    "mean_reward": round(sum(r["reward"] for r in eval_results) / n, 4),
    "benchmark_tasks": len(eval_results),
}
(final_adapter_dir / "eval_scorecard.json").write_text(json.dumps(scorecard, indent=2) + "\n")

registry_entry = {
    "id": f"community-{TRADE}-v1",
    "trade": TRADE,
    "hf_repo": HF_OUTPUT_REPO,
    "revision": "main",
    "contributor": "your-github-handle",
    "method": "grpo" if RUN_GRPO else "sft",
    "default_weight": -0.3 if TRADE == "clash_negation" else 0.8,
    "scorecard": scorecard,
}
print("Eval Scorecard:", json.dumps(scorecard, indent=2))
print("\nRegistry PR snippet for registry.json:\n", json.dumps(registry_entry, indent=2))

if PUSH_TO_HUB:
    from huggingface_hub import HfApi, login
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    login(token=token)
    api = HfApi()
    api.create_repo(HF_OUTPUT_REPO, exist_ok=True)
    api.upload_folder(folder_path=str(final_adapter_dir), repo_id=HF_OUTPUT_REPO)
    print(f"Pushed LoRA + scorecard to https://huggingface.co/{HF_OUTPUT_REPO}")
